# MetalDock Colab
**QM-Enhanced Metal-Organic Complex Docking via AutoDock4**

1. Run **Cell 1** (installs packages, then runtime restarts automatically)
2. Run **Cell 2** (launches the interactive UI)
3. Use the tabbed panel: Setup (ORCA) → Dock → Visualize → Results

In [ ]:
#@title Step 1: Install Dependencies (runtime restarts after this cell)
import subprocess, sys, os

# System packages
subprocess.run(['apt-get', 'update', '-qq'], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'autodock', 'autogrid', 'openbabel'], check=False)

# Python packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'py3Dmol', 'ipywidgets', 'biopython', 'numpy', 'scipy',
                'pandas', 'openbabel-wheel', 'rdkit', 'ase', 'pdb2pqr',
                'meeko'], check=False)

# Clone MetalDock
if not os.path.isdir('/content/MetalDock'):
    subprocess.run(['git', 'clone',
                    'https://github.com/MatthijsHak/MetalDock.git',
                    '/content/MetalDock'], check=False)
else:
    subprocess.run(['git', '-C', '/content/MetalDock', 'pull'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e',
                '/content/MetalDock'], check=False)

# Clone AcuDockMetal (shared utilities)
if not os.path.isdir('/content/AcuDockMetal'):
    subprocess.run(['git', 'clone',
                    'https://github.com/Grimlock5310/AcuDockMetal.git',
                    '/content/AcuDockMetal'], check=False)
else:
    subprocess.run(['git', '-C', '/content/AcuDockMetal', 'pull'], check=False)

# Restart runtime so native modules reload
os.kill(os.getpid(), 9)

In [ ]:
#@title Step 2: Launch Interactive UI

import os, sys, subprocess, configparser, urllib.request, warnings, glob, re, shutil, zipfile, tempfile, io
warnings.filterwarnings('ignore')

import ipywidgets as w
from IPython.display import display, HTML, clear_output

# ---------------------------------------------------------------------------
# State
# ---------------------------------------------------------------------------
_state = {
    'orca_path': None,
    'protein_pdb': None,
    'xyz_file': None,
    'workdir': None,
    'poses': [],       # list of dicts: {pdb_path, energy, rank}
}

METAL_SYMBOLS = {
    'Ru', 'Pt', 'Os', 'Ir', 'Pd', 'Fe', 'Co', 'Cu', 'Zn', 'Ni', 'Mn',
    'Rh', 'Re', 'Au', 'Ag', 'Cr', 'Mo', 'W', 'V', 'Ti',
}

# ---------------------------------------------------------------------------
# Header
# ---------------------------------------------------------------------------
header = w.HTML(
    '<h2 style="margin:0">MetalDock Colab</h2>'
    '<p style="color:#666;margin:0 0 8px">'
    'QM-Enhanced Metal-Organic Complex Docking via ORCA + AutoDock4</p>'
)

# ===================================================================
# Tab 0 — Setup (ORCA)
# ===================================================================
orca_info = w.HTML(
    '<b>ORCA Setup</b><br>'
    'ORCA is free for academics. Register at '
    '<a href="https://orcaforum.kofo.mpg.de" target="_blank">orcaforum.kofo.mpg.de</a>, '
    'download the Linux x86_64 tarball, and upload below.<br>'
    'Alternatively, enter the path if ORCA is already installed.'
)
orca_upload = w.FileUpload(accept='.tar.xz,.tar.gz,.tar.bz2,.zip',
                           description='Upload ORCA',
                           layout=w.Layout(width='360px'))
orca_path_txt = w.Text(description='Or ORCA path:',
                       placeholder='/content/orca/orca',
                       style={'description_width': '100px'},
                       layout=w.Layout(width='460px'))
orca_verify_btn = w.Button(description='Verify ORCA', button_style='info',
                           icon='check', layout=w.Layout(width='160px'))
orca_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px'))

def _extract_orca(change):
    orca_out.clear_output()
    with orca_out:
        if not orca_upload.value:
            print('No file selected.')
            return
        item = list(orca_upload.value.values())[0] if isinstance(orca_upload.value, dict) else orca_upload.value[0]
        content = item['content'] if isinstance(item, dict) else item.content
        name = item['name'] if isinstance(item, dict) else item.name
        dest = '/content/orca'
        os.makedirs(dest, exist_ok=True)
        print(f'Extracting {name}...')
        tmp = os.path.join(dest, name)
        with open(tmp, 'wb') as f:
            f.write(content)
        if name.endswith('.zip'):
            with zipfile.ZipFile(tmp) as zf:
                zf.extractall(dest)
        else:
            subprocess.run(['tar', 'xf', tmp, '-C', dest], check=True)
        os.remove(tmp)
        # Find orca binary
        for root, dirs, files in os.walk(dest):
            if 'orca' in files:
                orca_bin = os.path.join(root, 'orca')
                os.chmod(orca_bin, 0o755)
                orca_path_txt.value = orca_bin
                print(f'Found ORCA at {orca_bin}')
                return
        print('Warning: could not find orca binary in archive.')

orca_upload.observe(_extract_orca, names='value')

def _verify_orca(btn):
    orca_out.clear_output()
    with orca_out:
        path = orca_path_txt.value.strip()
        if not path:
            # Try to find on PATH
            result = shutil.which('orca')
            if result:
                path = result
                orca_path_txt.value = path
        if not path or not os.path.isfile(path):
            print('ORCA binary not found. Upload or enter the path.')
            return
        os.environ['PATH'] = os.path.dirname(path) + ':' + os.environ.get('PATH', '')
        os.environ['ASE_ORCA_COMMAND'] = path + ' PREFIX.inp > PREFIX.out'
        _state['orca_path'] = path
        print(f'ORCA verified at: {path}')

orca_verify_btn.on_click(_verify_orca)

setup_tab = w.VBox([orca_info, orca_upload, orca_path_txt,
                     orca_verify_btn, orca_out])

# ===================================================================
# Tab 1 — Dock
# ===================================================================
pdb_input   = w.Text(value='1T6E', description='PDB ID:',
                     placeholder='e.g. 1T6E',
                     style={'description_width': '120px'},
                     layout=w.Layout(width='360px'))
pdb_upload  = w.FileUpload(accept='.pdb', description='Or upload PDB',
                           layout=w.Layout(width='360px'))
xyz_upload  = w.FileUpload(accept='.xyz', description='Metal complex XYZ',
                           layout=w.Layout(width='360px'))
metal_dd    = w.Dropdown(options=['Ru', 'Pt', 'Os', 'Ir', 'Pd', 'Fe',
                                  'Co', 'Cu', 'Zn', 'Ni', 'Mn', 'Rh',
                                  'Re', 'Au', 'Ag'],
                         value='Ru', description='Metal:',
                         style={'description_width': '120px'},
                         layout=w.Layout(width='360px'))
charge_sl   = w.IntSlider(value=0, min=-4, max=4, description='Charge:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='360px'))
spin_sl     = w.IntSlider(value=0, min=0, max=6, description='Spin mult:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='360px'))
ph_sl       = w.FloatSlider(value=7.4, min=4.0, max=10.0, step=0.1,
                            description='pH:',
                            style={'description_width': '120px'},
                            layout=w.Layout(width='360px'))
orca_simple = w.Text(value='B3LYP def2-TZVP', description='ORCA input:',
                     style={'description_width': '120px'},
                     layout=w.Layout(width='360px'))
orca_blocks = w.Textarea(value='', description='ORCA blocks:',
                         rows=2, style={'description_width': '120px'},
                         layout=w.Layout(width='360px'))
dock_x      = w.FloatText(value=0.0, description='Box center X:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='260px'))
dock_y      = w.FloatText(value=0.0, description='Box center Y:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='260px'))
dock_z      = w.FloatText(value=0.0, description='Box center Z:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='260px'))
box_size_sl = w.IntSlider(value=20, min=10, max=60, description='Box size (A):',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='360px'))
num_poses_sl = w.IntSlider(value=10, min=1, max=50, description='Num poses:',
                           style={'description_width': '120px'},
                           layout=w.Layout(width='360px'))
ncpu_sl     = w.IntSlider(value=2, min=1, max=4, description='CPUs:',
                          style={'description_width': '120px'},
                          layout=w.Layout(width='360px'))
geom_opt_cb = w.Checkbox(value=True, description='Geometry optimization',
                         indent=False)
vacant_cb   = w.Checkbox(value=True, description='Vacant coordination site',
                         indent=False)
clean_pdb_cb = w.Checkbox(value=True, description='Clean PDB (rm HETATM)',
                          indent=False)

autodetect_btn = w.Button(description='Auto-detect center',
                          button_style='info', icon='crosshairs',
                          layout=w.Layout(width='180px'))
run_btn = w.Button(description='Run MetalDock', button_style='success',
                   icon='play', layout=w.Layout(width='180px', height='40px'))
dock_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px',
                                    max_height='500px', overflow_y='auto'))

def _get_pdb_path():
    """Return local PDB file path, downloading if needed."""
    if pdb_upload.value:
        item = list(pdb_upload.value.values())[0] if isinstance(pdb_upload.value, dict) else pdb_upload.value[0]
        content = item['content'] if isinstance(item, dict) else item.content
        name = item['name'] if isinstance(item, dict) else item.name
        path = os.path.join('/content', name)
        with open(path, 'wb') as f:
            f.write(content)
        return path
    pdb_id = pdb_input.value.strip().upper()
    path = f'/content/{pdb_id}.pdb'
    if not os.path.exists(path):
        print(f'Downloading {pdb_id}...')
        urllib.request.urlretrieve(
            f'https://files.rcsb.org/download/{pdb_id}.pdb', path)
    return path

def _get_xyz_path(workdir):
    """Return local XYZ file path from upload."""
    if not xyz_upload.value:
        return None
    item = list(xyz_upload.value.values())[0] if isinstance(xyz_upload.value, dict) else xyz_upload.value[0]
    content = item['content'] if isinstance(item, dict) else item.content
    name = item['name'] if isinstance(item, dict) else item.name
    path = os.path.join(workdir, name)
    with open(path, 'wb') as f:
        f.write(content)
    return path

def _autodetect_center(btn):
    dock_out.clear_output()
    with dock_out:
        try:
            from Bio.PDB import PDBParser
            pdb_path = _get_pdb_path()
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure('prot', pdb_path)
            metals = []
            for atom in structure.get_atoms():
                if atom.element and atom.element.strip().capitalize() in METAL_SYMBOLS:
                    metals.append(atom)
            if metals:
                m = metals[0]
                coord = m.get_coord()
                dock_x.value = round(float(coord[0]), 1)
                dock_y.value = round(float(coord[1]), 1)
                dock_z.value = round(float(coord[2]), 1)
                print(f'Detected {m.element.strip()} at '
                      f'({dock_x.value}, {dock_y.value}, {dock_z.value})')
                if len(metals) > 1:
                    print(f'  ({len(metals)} metals found, using first)')
            else:
                # Fallback: centroid of CA atoms
                cas = [a.get_coord() for a in structure.get_atoms()
                       if a.get_name() == 'CA']
                if cas:
                    import numpy as np
                    c = np.mean(cas, axis=0)
                    dock_x.value = round(float(c[0]), 1)
                    dock_y.value = round(float(c[1]), 1)
                    dock_z.value = round(float(c[2]), 1)
                    print(f'No metal found; using CA centroid '
                          f'({dock_x.value}, {dock_y.value}, {dock_z.value})')
        except Exception as e:
            print(f'Error: {e}')

autodetect_btn.on_click(_autodetect_center)

def _run_docking(btn):
    dock_out.clear_output()
    with dock_out:
        # Validate ORCA
        if not _state.get('orca_path'):
            print('Please configure ORCA first (Setup tab).')
            return
        # Validate XYZ
        if not xyz_upload.value:
            print('Please upload a metal complex XYZ file.')
            return

        pdb_id = pdb_input.value.strip().upper() or 'custom'
        workdir = f'/content/metaldock_run_{pdb_id}'
        os.makedirs(workdir, exist_ok=True)
        _state['workdir'] = workdir

        pdb_path = _get_pdb_path()
        _state['protein_pdb'] = pdb_path

        xyz_path = _get_xyz_path(workdir)
        _state['xyz_file'] = xyz_path

        # Write .ini config
        config = configparser.ConfigParser()
        config['DEFAULT'] = {
            'metal_symbol': metal_dd.value,
            'ncpu': str(ncpu_sl.value),
        }
        config['PROTEIN'] = {
            'pdb_file': os.path.abspath(pdb_path),
            'pH': str(ph_sl.value),
            'clean_pdb': str(clean_pdb_cb.value),
        }
        config['QM'] = {
            'engine': 'ORCA',
            'orcasimpleinput': orca_simple.value.strip(),
            'orcablocks': orca_blocks.value.strip(),
        }
        config['METAL_COMPLEX'] = {
            'geom_opt': str(geom_opt_cb.value),
            'xyz_file': os.path.abspath(xyz_path),
            'charge': str(charge_sl.value),
            'spin': str(spin_sl.value),
            'vacant_site': str(vacant_cb.value),
        }
        config['DOCKING'] = {
            'ga_dock': 'True',
            'dock_x': str(dock_x.value),
            'dock_y': str(dock_y.value),
            'dock_z': str(dock_z.value),
            'box_size': str(box_size_sl.value),
            'random_pos': 'True',
            'num_poses': str(num_poses_sl.value),
        }
        ini_path = os.path.join(workdir, 'metaldock_input.ini')
        with open(ini_path, 'w') as f:
            config.write(f)
        print(f'Config written to {ini_path}')

        # Run MetalDock
        print('Starting MetalDock...')
        proc = subprocess.Popen(
            ['metaldock', '-i', ini_path, '-m', 'dock'],
            cwd=workdir,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
        proc.wait()

        if proc.returncode != 0:
            print(f'\nMetalDock exited with code {proc.returncode}')
            return

        # Parse results — look for DLG files
        print('\nParsing results...')
        _state['poses'] = []
        dlg_files = glob.glob(os.path.join(workdir, '**', '*.dlg'),
                              recursive=True)
        for dlg in dlg_files:
            with open(dlg) as f:
                text = f.read()
            energies = re.findall(
                r'Estimated Free Energy of Binding\s*=\s*([-\d.]+)\s*kcal/mol',
                text)
            for i, e in enumerate(energies):
                _state['poses'].append({
                    'energy': float(e),
                    'rank': i + 1,
                    'dlg_path': dlg,
                })

        # Also look for result PDB/PDBQT files
        result_files = sorted(
            glob.glob(os.path.join(workdir, '**', '*.pdbqt'), recursive=True) +
            glob.glob(os.path.join(workdir, '**', '*pose*.pdb'), recursive=True))
        for i, rf in enumerate(result_files):
            if i < len(_state['poses']):
                _state['poses'][i]['pdb_path'] = rf
            else:
                _state['poses'].append({
                    'energy': 0.0, 'rank': i + 1, 'pdb_path': rf,
                })

        _state['poses'].sort(key=lambda p: p.get('energy', 0.0))
        for i, p in enumerate(_state['poses']):
            p['rank'] = i + 1

        n = len(_state['poses'])
        print(f'Done. Found {n} pose(s).')
        if _state['poses']:
            best = _state['poses'][0]
            print(f'Best energy: {best["energy"]:.1f} kcal/mol')

run_btn.on_click(_run_docking)

left_col = w.VBox([pdb_input, pdb_upload, xyz_upload, metal_dd,
                    charge_sl, spin_sl])
right_col = w.VBox([ph_sl, orca_simple, orca_blocks,
                     w.HBox([dock_x, dock_y, dock_z]),
                     box_size_sl, num_poses_sl, ncpu_sl])
options_row = w.HBox([geom_opt_cb, vacant_cb, clean_pdb_cb])
dock_tab = w.VBox([
    w.HBox([left_col, right_col]),
    options_row,
    w.HBox([autodetect_btn, run_btn]),
    dock_out,
])

# ===================================================================
# Tab 2 — Visualize
# ===================================================================
pose_dd = w.Dropdown(options=['(run docking first)'], description='Pose:',
                     style={'description_width': '50px'},
                     layout=w.Layout(width='300px'))
show_btn = w.Button(description='Show', button_style='primary', icon='eye',
                    layout=w.Layout(width='120px'))
table_btn = w.Button(description='Score Table', button_style='info',
                     icon='table', layout=w.Layout(width='140px'))
viz_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px',
                                   min_height='400px'))

def _refresh_pose_dd():
    if _state['poses']:
        pose_dd.options = [
            f"Pose {p['rank']}: {p['energy']:.1f} kcal/mol"
            for p in _state['poses']
        ]
    else:
        pose_dd.options = ['(run docking first)']

def _show_pose(btn):
    viz_out.clear_output()
    _refresh_pose_dd()
    with viz_out:
        if not _state['poses']:
            print('Run docking first (Dock tab).')
            return
        import py3Dmol
        idx = pose_dd.index if pose_dd.index is not None else 0
        pose = _state['poses'][idx]
        pose_path = pose.get('pdb_path')
        prot_path = _state.get('protein_pdb')

        if not prot_path or not os.path.isfile(prot_path):
            print('Protein PDB not found.')
            return

        viewer = py3Dmol.view(width=800, height=500)

        # Protein
        with open(prot_path) as f:
            viewer.addModel(f.read(), 'pdb')
        viewer.setStyle({'model': 0},
                        {'cartoon': {'color': 'spectrum', 'opacity': 0.7}})

        # Docked pose
        if pose_path and os.path.isfile(pose_path):
            fmt = 'pdb' if pose_path.endswith('.pdb') else 'pdbqt'
            with open(pose_path) as f:
                viewer.addModel(f.read(), fmt)
            viewer.setStyle({'model': 1}, {'stick': {'radius': 0.15}})
            # Highlight metal as sphere
            metal = metal_dd.value
            viewer.setStyle(
                {'model': 1, 'elem': metal},
                {'sphere': {'radius': 0.8, 'color': '#E06633'}})
            viewer.zoomTo({'model': 1})
        else:
            print(f'Pose file not found for pose {idx+1}.')
            viewer.zoomTo()

        viewer.show()

def _show_table(btn):
    viz_out.clear_output()
    _refresh_pose_dd()
    with viz_out:
        if not _state['poses']:
            print('Run docking first (Dock tab).')
            return
        import pandas as pd
        rows = [{'Rank': p['rank'], 'Energy (kcal/mol)': p['energy']}
                for p in _state['poses']]
        df = pd.DataFrame(rows)
        display(HTML(df.to_html(index=False)))

show_btn.on_click(_show_pose)
table_btn.on_click(_show_table)

viz_tab = w.VBox([w.HBox([pose_dd, show_btn, table_btn]), viz_out])

# ===================================================================
# Tab 3 — Results
# ===================================================================
results_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px'))
refresh_btn = w.Button(description='Refresh', button_style='info',
                       icon='refresh', layout=w.Layout(width='120px'))
download_btn = w.Button(description='Download All (ZIP)',
                        button_style='success', icon='download',
                        layout=w.Layout(width='200px'))

def _refresh_results(btn):
    results_out.clear_output()
    with results_out:
        wd = _state.get('workdir')
        if not wd or not os.path.isdir(wd):
            print('No results yet. Run docking first.')
            return
        print(f'Output directory: {wd}\n')
        for root, dirs, files in os.walk(wd):
            level = root.replace(wd, '').count(os.sep)
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/')
            sub_indent = '  ' * (level + 1)
            for f in sorted(files):
                fpath = os.path.join(root, f)
                size = os.path.getsize(fpath)
                if size > 1024 * 1024:
                    sz = f'{size / 1024 / 1024:.1f} MB'
                elif size > 1024:
                    sz = f'{size / 1024:.1f} KB'
                else:
                    sz = f'{size} B'
                print(f'{sub_indent}{f}  ({sz})')

def _download_all(btn):
    results_out.clear_output()
    with results_out:
        wd = _state.get('workdir')
        if not wd or not os.path.isdir(wd):
            print('No results to download.')
            return
        zip_path = f'{wd}.zip'
        print(f'Creating {zip_path}...')
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(wd):
                for f in files:
                    fpath = os.path.join(root, f)
                    arcname = os.path.relpath(fpath, os.path.dirname(wd))
                    zf.write(fpath, arcname)
        print(f'ZIP created: {zip_path}')
        try:
            from google.colab import files as colab_files
            colab_files.download(zip_path)
        except ImportError:
            print('(Not running on Colab — download manually)')

refresh_btn.on_click(_refresh_results)
download_btn.on_click(_download_all)

results_tab = w.VBox([w.HBox([refresh_btn, download_btn]), results_out])

# ===================================================================
# Assemble Tabs
# ===================================================================
tabs = w.Tab(children=[setup_tab, dock_tab, viz_tab, results_tab])
tabs.set_title(0, 'Setup (ORCA)')
tabs.set_title(1, 'Dock')
tabs.set_title(2, 'Visualize')
tabs.set_title(3, 'Results')

display(header, tabs)